<a href="https://colab.research.google.com/github/Isi2000/NLP/blob/main/TRANS.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!git clone https://github.com/Isi2000/NLP.git

import math
import os
from typing import Tuple
from tempfile import TemporaryDirectory

import torch
from torch import nn, Tensor
from torch.nn import TransformerEncoder, TransformerEncoderLayer
from torch.utils.data import dataset

!pip install Bio
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from Bio import SeqIO
import string
import random
from tqdm import tqdm

Cloning into 'NLP'...
remote: Enumerating objects: 6, done.
remote: Counting objects: 100% (6/6), done.
remote: Compressing objects: 100% (4/4), done.
remote: Total 6 (delta 0), reused 6 (delta 0), pack-reused 0
Receiving objects: 100% (6/6), 4.27 MiB | 6.71 MiB/s, done.
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.6/278.6 kB 4.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 13.4 MB/s eta 0:00:00


In [ ]:

class TransformerModel(nn.Module):
    def __init__(self, ntoken: int,
                 d_model: int,
                 nhead: int,
                 d_hid: int,
                 nlayers: int,
                 dropout: float = 0.2):

        """
        Docstring since you are new in this field
        ntoken: num of tokens in my vocab
        d_model: dim of the input and output embedding
        nhead: number of attention heads in multiattention model
        d_hid: dim of hidden layers
        nlayers: number of layers in trans
        dropout: self explanatory
        """
        super().__init__()
        self.model_type = 'Transformer'
        self.pos_encoder = PositionalEncoding(d_model, dropout) #defined later
        self.encode_layers = TransformerEncoderLayer(d_model, nhead, d_hid, dropout) #encoder
        self.transformer_encoder = TransformerEncoder(self.encode_layers, nlayers) #encoder
        self.embedding = nn.Embedding(ntoken, d_model) #my embedding
        self.d_model = d_model # doc
        self.linear = nn.Linear(d_model, ntoken) #liear state

        self.init_weights() #defined later

    def init_weights(self) -> "inizializzo i pesi":
        """
        this class initialized the weigths:
        for the embedding as uniform in the range
        for linear as uniform in the range with bias == 0
        """
        initrange = 0.1
        self.embedding.weight.data.uniform_(-initrange, initrange)
        self.linear.bias.data.zero_()
        self.linear.weight.data.uniform_(-initrange, initrange)

    def forward(self, src: Tensor,
                src_mask: Tensor = None):
        """
        Arguments:
            src: Tensor, shape ``[seq_len, batch_size]``
            src_mask: Tensor, shape ``[seq_len, seq_len]``

        Returns:
            output Tensor of shape ``[seq_len, batch_size,
        """
        src = self.embedding(src) * math.sqrt(self.d_model)
        src = self.pos_encoder(src)
        if src_mask is None:
            """Generate a square causal mask for the sequence. The masked positions are filled with float('-inf').
            Unmasked positions are filled with float(0.0).
            """
            src_mask = nn.Transformer.generate_square_subsequent_mask(len(src)).to(device)
        output = self.transformer_encoder(src, src_mask)
        output = self.linear(output)
        return output


class PositionalEncoding(nn.Module):
    #questa classe fa roba molto poco comprensibile per encodare
    def __init__(self, d_model: int, dropout: float = 0.1, max_len: int = 5000):
        super().__init__()
        self.dropout = nn.Dropout(p=dropout)

        position = torch.arange(max_len).unsqueeze(1)
        div_term = torch.exp(torch.arange(0, d_model, 2) * (-math.log(10000.0) / d_model))
        pe = torch.zeros(max_len, 1, d_model)
        pe[:, 0, 0::2] = torch.sin(position * div_term)
        pe[:, 0, 1::2] = torch.cos(position * div_term)
        self.register_buffer('pe', pe) #non si considera quando si fa il grad

    def forward(self, x: Tensor) -> Tensor:
        """
        Arguments:
            x: Tensor, shape ``[seq_len, batch_size, embedding_dim]``
        """
        x = x + self.pe[:x.size(0)]
        return self.dropout(x)

### Il modello e' fatto, ora bisogna applicarlo ai dati


records = list(SeqIO.parse("/content/NLP/ird_influenzaA_HA_allspecies.fa", "fasta"))
ids = []
sequences = []
# Iterate over the records and extract data
for record in records:
    ids.append(record.id)
    sequences.append(str(record.seq))  # Convert the sequence to a string

df = pd.DataFrame({"ID": ids, "Sequence": sequences})


from torchtext.vocab import build_vocab_from_iterator

#se in un futuro vuoi passare ai birammi allora ci si pensa (non hai messo <unk>)
amino_a = ['A', 'B','R', 'N', 'D', 'C', 'Q', 'E', 'G', 'H', 'I',
               'L', 'K', 'M', 'F', 'P', 'S', 'T', 'W', 'Y', 'V', 'X', 'J']
vocab = {}
for idx, a in enumerate(amino_a):
    vocab[a] = idx
#print(vocab)


###QUESTO POOTREBBE DARMI DEI PROBLEMI


red_df = df.head(1000)
s = ""
for i,j in red_df.iterrows():
    s += j["Sequence"]

sequence_indices = [vocab[amino] for amino in s]
# Convert the sequence indices to a torch tensor
sequence_tensor = torch.tensor(sequence_indices, dtype=torch.long)
print(sequence_tensor)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

def batchify(data: Tensor, bsz: int) -> Tensor:
    """Divides the data into ``bsz`` separate sequences, removing extra elements
    that wouldn't cleanly fit.

    Arguments:
        data: Tensor, shape ``[N]``
        bsz: int, batch size

    Returns:
        Tensor of shape ``[N // bsz, bsz]``
    """
    seq_len = data.size(0) // bsz
    data = data[:seq_len * bsz]
    data = data.view(bsz, seq_len).t().contiguous()
    return data.to(device)


n = int(len(sequence_tensor)*0.8)
train_data =  sequence_tensor[:n]
val_data = sequence_tensor[n:]

#SE NON FUNZIA QUA C'E' RISCHIO DI FARE DEL BORDELLO
batch_size = 20
eval_batch_size = 10
train_data = batchify(train_data, batch_size)  # shape ``[seq_len, batch_size]``
val_data = batchify(val_data, eval_batch_size)
print(len(train_data[0]))


bptt = 35
def get_batch(source: Tensor, i: int) -> Tuple[Tensor, Tensor]:
    """
    Args:
        source: Tensor, shape ``[full_seq_len, batch_size]``
        i: int

    Returns:
        tuple (data, target), where data has shape ``[seq_len, batch_size]`` and
        target has shape ``[seq_len * batch_size]``
    """
    seq_len = min(bptt, len(source) - 1 - i)
    data = source[i:i+seq_len]
    target = source[i+1:i+1+seq_len].reshape(-1)
    return data, target

ntokens = len(vocab)  # size of vocabulary
emsize = 200  # embedding dimension
d_hid = 200  # dimension of the feedforward network model in ``nn.TransformerEncoder``
nlayers = 2  # number of ``nn.TransformerEncoderLayer`` in ``nn.TransformerEncoder``
nhead = 2  # number of heads in ``nn.MultiheadAttention``
dropout = 0.2  # dropout probability

model = TransformerModel(ntokens, emsize, nhead, d_hid, nlayers, dropout)#.to(device)




tensor([13,  3, 17,  ..., 10,  5, 10])
20


/usr/local/lib/python3.10/dist-packages/torch/nn/modules/transformer.py:282: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.self_attn.batch_first was not True(use batch_first for better inference performance)
  warnings.warn(f"enable_nested_tensor is True, but self.use_nested_tensor is False because {why_not_sparsity_fast_path}")


In [ ]:
!pip install torchviz
import torch
from torchviz import make_dot

# Assuming ntoken is the size of your vocabulary
ntoken = 1000
# Assuming d_model is the dimension of input and output embedding
d_model = 512
# Assuming you have other parameters needed for initialization
nhead = 8
d_hid = 2048
nlayers = 6
dropout = 0.2

# Create a sample input tensor (you need to adjust this based on your input shape)
# Here, I'm creating a tensor with shape [seq_len, batch_size]
sample_input = torch.randint(ntoken, (20, 1))  # Adjust seq_len and batch_size according to your input shape

# Instantiate your model
model = TransformerModel(ntoken, d_model, nhead, d_hid, nlayers, dropout)

# Generate predictions for the sample data
output = model(sample_input)

# Generate a model architecture visualization
make_dot(output.mean(),
         params=dict(model.named_parameters()),
         show_attrs=True,
         show_saved=True).render("TransformerModel_torchviz", format="png")


  Preparing metadata (setup.py) ... done
  Created wheel for torchviz: filename=torchviz-0.0.2-py3-none-any.whl size=4131 sha256=ce2f5d8c20963ba6cd55793a94b92716ba62eca807cdb21fe5a2b5961b07dbf0
  Stored in directory: /root/.cache/pip/wheels/4c/97/88/a02973217949e0db0c9f4346d154085f4725f99c4f15a87094
Successfully built torchviz


/usr/local/lib/python3.10/dist-packages/torch/nn/modules/transformer.py:282: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.self_attn.batch_first was not True(use batch_first for better inference performance)
  warnings.warn(f"enable_nested_tensor is True, but self.use_nested_tensor is False because {why_not_sparsity_fast_path}")


'TransformerModel_torchviz.png'

In [ ]:
###QUESTA PARTE LA DEVI RIVEDERE SICURO

import time

criterion = nn.CrossEntropyLoss()
lr = 5.0  # learning rate
optimizer = torch.optim.SGD(model.parameters(), lr=lr)
scheduler = torch.optim.lr_scheduler.StepLR(optimizer, 1.0, gamma=0.95)

def train(model: nn.Module) -> None:
    model.train()  # turn on train mode
    total_loss = 0.
    log_interval = 200
    start_time = time.time()

    num_batches = len(train_data) // bptt
    for batch, i in enumerate(range(0, train_data.size(0) - 1, bptt)):
        data, targets = get_batch(train_data, i)
        output = model(data)
        output_flat = output.view(-1, ntokens)
        loss = criterion(output_flat, targets)

        optimizer.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 0.5)
        optimizer.step()

        total_loss += loss.item()
        if batch % log_interval == 0 and batch > 0:
            lr = scheduler.get_last_lr()[0]
            ms_per_batch = (time.time() - start_time) * 1000 / log_interval
            cur_loss = total_loss / log_interval
            ppl = math.exp(cur_loss)
            print(f'| epoch {epoch:3d} | {batch:5d}/{num_batches:5d} batches | '
                  f'lr {lr:02.2f} | ms/batch {ms_per_batch:5.2f} | '
                  f'loss {cur_loss:5.2f} | ppl {ppl:8.2f}')
            total_loss = 0
            start_time = time.time()

def evaluate(model: nn.Module, eval_data: Tensor) -> float:
    model.eval()  # turn on evaluation mode
    total_loss = 0.
    with torch.no_grad():
        for i in range(0, eval_data.size(0) - 1, bptt):
            data, targets = get_batch(eval_data, i)
            seq_len = data.size(0)
            output = model(data)
            output_flat = output.view(-1, ntokens)
            total_loss += seq_len * criterion(output_flat, targets).item()
    return total_loss / (len(eval_data) - 1)

best_val_loss = float('inf')
epochs = 10

with TemporaryDirectory() as tempdir:
    best_model_params_path = os.path.join(tempdir, "best_model_params.pt")

    for epoch in range(1, epochs + 1):
        epoch_start_time = time.time()
        train(model)
        val_loss = evaluate(model, val_data)
        val_ppl = math.exp(val_loss)
        elapsed = time.time() - epoch_start_time
        print('-' * 89)
        print(f'| end of epoch {epoch:3d} | time: {elapsed:5.2f}s | '
            f'valid loss {val_loss:5.2f} | valid ppl {val_ppl:8.2f}')
        print('-' * 89)

        if val_loss < best_val_loss:
            best_val_loss = val_loss
            torch.save(model.state_dict(), best_model_params_path)

        scheduler.step()
    model.load_state_dict(torch.load(best_model_params_path)) # load best model states


In [ ]:
import torch.nn.functional as F

def generate_next_token(output):
    # Apply softmax to convert logits into probabilities
    probabilities = F.softmax(output, dim=-1)
    # Sample from the probabilities to generate the next token
    next_token_index = torch.multinomial(probabilities.squeeze(), 1)
    # Convert the index back to the token using the vocabulary

output = model(train_data[0][0])
print(train_data[0][0])
print(len(output))
print(f"The predicted next token is: {next_token}")
